In [2]:
import pandas as pd
import numpy as np

# --- 1. INITIAL SETUP (MUST BE UPDATED) ---

# PLEASE PROVIDE THE EXACT PATH TO YOUR CLEANED, TRIP-LEVEL DATA FILE
DATA_PATH = 'YOUR_CLEANED_TRIP_DATA_PATH_HERE' 
# Example based on your previous notebook: DATA_PATH = r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet'

# Analysis Parameters
Z_SCORE_THRESHOLD = 3 

# Required Column Names (Ensuring consistency with your notebook's column names)
COL_DATETIME = 'tpep_pickup_datetime'
COL_DURATION = 'trip_duration_minutes' # Unit: minutes
COL_SPEED = 'trip_speed_mph' # Unit: mph

# ---------------------------------------------

def load_data(path: str) -> pd.DataFrame:
    """Loads the dataset and performs basic datetime standardization."""
    if path.endswith('.parquet'):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)
        
    df[COL_DATETIME] = pd.to_datetime(df[COL_DATETIME])
    return df

def calculate_contextual_z_score(df: pd.DataFrame, analyze_col: str, z_threshold: float) -> pd.DataFrame:
    """
    Calculates the Z-score relative to the 'Hour of Day x Day of Week' context.
    This prevents comparing an 3 AM trip to the 8 AM rush hour average.
    """
    
    # Create temporal context features
    df['day_of_week'] = df[COL_DATETIME].dt.dayofweek # 0=Monday
    df['hour'] = df[COL_DATETIME].dt.hour
    
    # 1. Calculate Group Statistics (Mean and Std Dev for 168 groups)
    group_keys = ['day_of_week', 'hour']
    
    stats = df.groupby(group_keys)[analyze_col].agg(['mean', 'std'])
    stats.columns = [f'{analyze_col}_mean', f'{analyze_col}_std']
    
    # 2. Merge statistics and calculate Z-score
    df = df.merge(stats, on=group_keys, how='left')
    
    col_std = f'{analyze_col}_std'
    col_z = f'Z_{analyze_col}'
    col_outlier = f'Outlier_{analyze_col}'
    
    # Z-score Formula: (Value - Mean) / Standard Deviation
    # Handle std=0 by replacing with 1 to prevent division by zero errors
    df[col_z] = (df[analyze_col] - df[f'{analyze_col}_mean']) / df[col_std].replace(0, 1)
    
    # 3. Mark the Outliers (|Z| > Threshold)
    df[col_outlier] = abs(df[col_z]) > z_threshold
    
    return df

# --- RUN MAIN PROGRAM ---
if __name__ == "__main__":
    try:
        # Load the data and update the path here
        DATA_PATH = r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet' # CONFIRMED PATH
        df_trips = load_data(DATA_PATH)
        print(f"Starting Z-score analysis on {len(df_trips)} trips.")

        # Analyze Z-score for Duration and Speed
        df_trips = calculate_contextual_z_score(df_trips, COL_DURATION, Z_SCORE_THRESHOLD)
        df_trips = calculate_contextual_z_score(df_trips, COL_SPEED, Z_SCORE_THRESHOLD)

        # --- CONSOLE REPORTING FOR TECHNICAL REPORT ---
        
        # 1. Total Outliers
        total_unique_outliers = df_trips[
            df_trips[f'Outlier_{COL_DURATION}'] | df_trips[f'Outlier_{COL_SPEED}']
        ].shape[0]
        
        print("\n--- Z-SCORE ANALYSIS RESULTS FOR REPORT ---")
        print(f"1. Total unique outlier trips (|Z| > {Z_SCORE_THRESHOLD}): {total_unique_outliers}")
        print(f"2. Outlier ratio: {total_unique_outliers / len(df_trips):.2%}")
        
        # 2. Detailed Classification
        print("\n3. Detailed Outlier Classification (Z > 3):")
        detailed_outliers = {
            "Duration: Too Long": df_trips[df_trips[f'Z_{COL_DURATION}'] > Z_SCORE_THRESHOLD].shape[0],
            "Duration: Too Short": df_trips[df_trips[f'Z_{COL_DURATION}'] < -Z_SCORE_THRESHOLD].shape[0],
            "Speed: Too Fast": df_trips[df_trips[f'Z_{COL_SPEED}'] > Z_SCORE_THRESHOLD].shape[0],
            "Speed: Too Slow": df_trips[df_trips[f'Z_{COL_SPEED}'] < -Z_SCORE_THRESHOLD].shape[0]
        }
        for k, v in detailed_outliers.items():
            print(f"- {k}: {v}")

        # 3. Listing Extreme Cases (for Narrative Interpretation)
        cols_for_display = [COL_DATETIME, 'trip_distance', COL_DURATION, COL_SPEED, f'Z_{COL_DURATION}', f'Z_{COL_SPEED}']
        
        print("\n4. Extreme Case Example (Speed: Too Slow, Z < -5):")
        extreme_slow = df_trips[df_trips[f'Z_{COL_SPEED}'] < -5].sort_values(by=f'Z_{COL_SPEED}').head(1)
        if not extreme_slow.empty:
            print(extreme_slow[cols_for_display])
        
    except FileNotFoundError:
        print(f"\n!!! ERROR: DATA FILE NOT FOUND AT {DATA_PATH}. Please ensure the notebook 4-1 was run.")
    except Exception as e:
        print(f"\nAn error occurred during data processing: {e}")


An error occurred during data processing: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
